# Lenguajes y Autómatas II (SCD-1016)
## Sesión 22 · Generación de TAC para Expresiones Aritmético-Lógicas

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo se traduce un árbol de expresión a una lista plana de instrucciones sin perder la precedencia?
>
> **Objetivo de la sesión:** Recorrer un AST en postorden y emitir una instrucción TAC por cada operador, devolviendo el nombre del temporal o de la hoja que contiene el resultado.

**Entregable:** tu función generar_tac() en artesano_sesion22_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion22.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes la raíz de un árbol con hojas y operadores binarios. Debes escribir una línea TAC por operador en el arreglo lineas y devolver cuántas líneas emitiste; las hojas no generan instrucción, solo prestan su nombre.

Al final de la sesión tendrás: **Un traductor en C que convierte un árbol binario en líneas TAC y respeta la precedencia con temporales _t0, _t1, ...**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 22](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion22.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion22.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion22.h"

int generar_tac(const Nodo *raiz, char lineas[][64], int max);
```

- Primero se calculan los hijos y después se emite el operador: el recorrido es postorden.
- Cada llamada empieza en _t0 y no reutiliza un temporal mientras siga vivo.
- Nunca escribas más de max líneas ni más de 31 caracteres por línea.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion22_c.c` junto con `artesano_sesion22_c.c`
> (o `ia_sesion22_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `a + b * c` | `_t0 = b * c` |
| 2 | `a + b * c (2a linea)` | `_t1 = a + _t0` |
| 3 | `(a + b) * c` | `_t1 = _t0 * c` |
| 4 | `x * (y - z) + w` | `_t2 = _t1 + w` |
| 5 | `hoja sin operaciones` | 0 |
| 6 | `(a + b) * (c - d)` | `_t2 = _t0 * _t1` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 22 - Traductor de AST a TAC con precedencia ===
Caso                               esperado:              obtenido:              resultado
[ 1] a + b * c                          esperado: _t0 = b * c  obtenido: _t0 = b * c  OK
[ 2] a + b * c (2a linea)
     esperado: _t1 = a + _t0
     obtenido: _t1 = a + _t0
     -> OK
[ 3] (a + b) * c
     esperado: _t1 = _t0 * c
     obtenido: _t1 = _t0 * c
     -> OK
[ 4] x * (y - z) + w
     esperado: _t2 = _t1 + w
     obtenido: _t2 = _t1 + w
     -> OK
[ 5] hoja sin operaciones               esperado: 0            obtenido: 0            OK
[ 6] (a + b) * (c - d)
     esperado: _t2 = _t0 * _t1
     obtenido: _t2 = _t0 * _t1
     -> OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion22.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] Cada operador genera exactamente una línea y las hojas no generan ninguna.
- [ ] El orden de las líneas respeta la precedencia del árbol, no el orden de lectura.
- [ ] Los seis casos del harness salen OK con código de salida 0.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion22_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion22.h
/* api_sesion22.h - contrato de la Sesion 22
 * Traductor de AST a TAC con precedencia
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION22_H
#define API_SESION22_H

typedef struct Nodo {
    char tipo;
    char op;
    char valor[16];
    struct Nodo *izq;
    struct Nodo *der;
} Nodo;

int generar_tac(const Nodo *raiz, char lineas[][64], int max);

#endif /* API_SESION22_H */


In [ ]:
%%writefile artesano_sesion22_c.c
/* artesano_sesion22_c.c - Fase 1: El Artesano (sin IA)
 * Traductor de AST a TAC con precedencia.
 *
 * Implementa lo que declara api_sesion22.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion22.h"
#include <stdio.h>
#include <string.h>

int generar_tac(const Nodo *raiz, char lineas[][64], int max) {
          (void)raiz; (void)lineas; (void)max; /* TODO: postorden y temporales */
          return 0;
      }


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion22_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion22_c.c
/* ia_sesion22_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion22.h. No puede traer main().
 */
#include "api_sesion22.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion22_c.c
/* harness_sesion22_c.c - auditoria automatica de la Sesion 22
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion22_c.c artesano_sesion22_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion22.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_str_largo(const char *caso, const char *esperado,
                                const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %s\n", probadas, caso);
    printf("     esperado: %s\n", esperado);
    printf("     obtenido: %s\n", obtenido ? obtenido : "(NULL)");
    printf("     -> %s\n", ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 22 - Traductor de AST a TAC con precedencia ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { Nodo a = {.tipo='H', .valor="a"}; Nodo b = {.tipo='H', .valor="b"}; Nodo c = {.tipo='H', .valor="c"}; Nodo m = {.tipo='B', .op='*', .izq=&b, .der=&c}; Nodo s = {.tipo='B', .op='+', .izq=&a, .der=&m}; char l[8][64] = {{0}}; generar_tac(&s, l, 8); verificar_str("a + b * c", "_t0 = b * c", l[0]); }
    { Nodo a = {.tipo='H', .valor="a"}; Nodo b = {.tipo='H', .valor="b"}; Nodo c = {.tipo='H', .valor="c"}; Nodo m = {.tipo='B', .op='*', .izq=&b, .der=&c}; Nodo s = {.tipo='B', .op='+', .izq=&a, .der=&m}; char l[8][64] = {{0}}; generar_tac(&s, l, 8); verificar_str_largo("a + b * c (2a linea)", "_t1 = a + _t0", l[1]); }
    { Nodo a = {.tipo='H', .valor="a"}; Nodo b = {.tipo='H', .valor="b"}; Nodo c = {.tipo='H', .valor="c"}; Nodo s = {.tipo='B', .op='+', .izq=&a, .der=&b}; Nodo m = {.tipo='B', .op='*', .izq=&s, .der=&c}; char l[8][64] = {{0}}; generar_tac(&m, l, 8); verificar_str_largo("(a + b) * c", "_t1 = _t0 * c", l[1]); }
    { Nodo x = {.tipo='H', .valor="x"}; Nodo y = {.tipo='H', .valor="y"}; Nodo z = {.tipo='H', .valor="z"}; Nodo w = {.tipo='H', .valor="w"}; Nodo r = {.tipo='B', .op='-', .izq=&y, .der=&z}; Nodo m = {.tipo='B', .op='*', .izq=&x, .der=&r}; Nodo s = {.tipo='B', .op='+', .izq=&m, .der=&w}; char l[8][64] = {{0}}; generar_tac(&s, l, 8); verificar_str_largo("x * (y - z) + w", "_t2 = _t1 + w", l[2]); }
    { Nodo a = {.tipo='H', .valor="x"}; char l[4][64] = {{0}}; int n = generar_tac(&a, l, 4); verificar_int("hoja sin operaciones", (long)(0), (long)(n)); }
    { Nodo a = {.tipo='H', .valor="a"}; Nodo b = {.tipo='H', .valor="b"}; Nodo c = {.tipo='H', .valor="c"}; Nodo d = {.tipo='H', .valor="d"}; Nodo s1 = {.tipo='B', .op='+', .izq=&a, .der=&b}; Nodo s2 = {.tipo='B', .op='-', .izq=&c, .der=&d}; Nodo m = {.tipo='B', .op='*', .izq=&s1, .der=&s2}; char l[8][64] = {{0}}; generar_tac(&m, l, 8); verificar_str_largo("(a + b) * (c - d)", "_t2 = _t0 * _t1", l[2]); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion22.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion22_c.c artesano_sesion22_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion22_c.c ia_sesion22_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion22.md` a `auditorias/auditoria_sesion22.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
